In [ ]:
from google.colab import auth, drive
import json
from google.cloud import bigquery

# Авторизация в Google Cloud
auth.authenticate_user()

# Подключаем Google Диск (чтобы файл не удалился после закрытия сессии)
drive.mount('/content/drive')

# Настройки проекта
PROJECT_ID = 'educationalflutterapp-94f27'
TABLE_ID = f"{PROJECT_ID}.ru_ids.ru1" # Ваша таблица с данными
OUTPUT_PATH = '/content/drive/MyDrive/patents_ru_dataset1.jsonl'

client = bigquery.Client(project=PROJECT_ID)


Mounted at /content/drive


In [ ]:
print(f"Начинаю выгрузку из {TABLE_ID}...")

# Получаем итератор строк
rows = client.list_rows(TABLE_ID)

count = 0
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    for row in rows:
        # Превращаем строку Row в словарь (dict)
        record = dict(row.items())

        # Записываем в формате JSONL (одна строка - один объект)
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

        count += 1
        if count % 10000 == 0:
            print(f"Обработано {count} патентов...")

print(f"Успех! Файл сохранен на ваш Google Диск: {OUTPUT_PATH}")
print(f"Итого выгружено строк: {count}")


Начинаю выгрузку из educationalflutterapp-94f27.ru_ids.ru1...
Обработано 10000 патентов...
Обработано 20000 патентов...
Обработано 30000 патентов...
Обработано 40000 патентов...
Обработано 50000 патентов...
Обработано 60000 патентов...
Обработано 70000 патентов...
Обработано 80000 патентов...
Обработано 90000 патентов...
Обработано 100000 патентов...
Обработано 110000 патентов...
Обработано 120000 патентов...
Обработано 130000 патентов...
Обработано 140000 патентов...
Обработано 150000 патентов...
Обработано 160000 патентов...
Обработано 170000 патентов...
Обработано 180000 патентов...
Обработано 190000 патентов...
Обработано 200000 патентов...
Обработано 210000 патентов...
Обработано 220000 патентов...
Обработано 230000 патентов...
Обработано 240000 патентов...
Обработано 250000 патентов...
Обработано 260000 патентов...
Обработано 270000 патентов...
Обработано 280000 патентов...
Обработано 290000 патентов...
Обработано 300000 патентов...
Обработано 310000 патентов...
Обработано 320000

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

# Смотрим что есть на Drive
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.endswith('.jsonl'):
            print(os.path.join(root, file))

/content/drive/MyDrive/patents_ru_dataset.jsonl
/content/drive/MyDrive/patents_ru_dataset1.jsonl


In [ ]:
import json

# Загружаем первый файл (основной - с abstract и citation)
file1 = {}
with open("/content/drive/MyDrive/patents_ru_dataset.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        p = json.loads(line)
        file1[p["publication_number"]] = p

print(f"Файл 1: {len(file1):,} патентов")

# Загружаем второй файл (с title, ipc_codes, family_id)
file2 = {}
with open("/content/drive/MyDrive/patents_ru_dataset1.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        p = json.loads(line)
        file2[p["publication_number"]] = p

print(f"Файл 2: {len(file2):,} патентов")
print(f"Совпадений по номеру: {len(set(file1) & set(file2)):,}")

Файл 1: 1,238,594 патентов
Файл 2: 1,238,525 патентов
Совпадений по номеру: 1,238,525


In [ ]:
# Объединяем
merged = []
for pub_num, patent in file1.items():
    extra = file2.get(pub_num, {})
    patent["title"] = extra.get("title", "")
    patent["ipc_codes"] = extra.get("ipc_codes", "")
    patent["family_id"] = extra.get("family_id", "")
    merged.append(patent)

print(f"Всего патентов: {len(merged):,}")
print(f"С title:        {sum(1 for p in merged if p.get('title')):,}")
print(f"С ipc_codes:    {sum(1 for p in merged if p.get('ipc_codes')):,}")
print(f"С abstract:    {sum(1 for p in merged if (p.get('abstract_ru') or '').strip()):,}")
print(f"С citation RU: {sum(1 for p in merged if any(c.get('publication_number','').startswith('RU') for c in (p.get('citation') or []))):,}")

Всего патентов: 1,238,594
С title:        1,238,525
С ipc_codes:    1,238,511
С abstract:    1,227,955
С citation RU: 548,805


In [ ]:
# Сохраняем на Google Drive
output_path = "/content/drive/MyDrive/patents_merged.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for p in merged:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

print(f"Сохранено → {output_path}")

Сохранено → /content/drive/MyDrive/patents_merged.jsonl


In [ ]:
import json
import random
from collections import defaultdict

# Загружаем merged файл
patents = {}
with open("/content/drive/MyDrive/patents_merged.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        p = json.loads(line)
        # Берём только патенты с рефератом
        if (p.get("abstract_ru") or "").strip():
            patents[p["publication_number"]] = p

print(f"Патентов с рефератом: {len(patents):,}")

# ── Позитивные пары из цитирований ──
citation_pairs = []
for pub_num, patent in patents.items():
    for c in (patent.get("citation") or []):
        cited = c.get("publication_number", "")
        if cited.startswith("RU") and cited in patents:
            pair = tuple(sorted([pub_num, cited]))
            citation_pairs.append(pair)

citation_pairs = list(set(citation_pairs))
print(f"Пар из цитирований:   {len(citation_pairs):,}")

# ── Позитивные пары из family_id ──
family_groups = defaultdict(list)
for pub_num, patent in patents.items():
    fid = patent.get("family_id", "")
    if fid:
        family_groups[fid].append(pub_num)

family_pairs = []
for fid, members in family_groups.items():
    if len(members) < 2:
        continue
    for i in range(len(members)):
        for j in range(i + 1, len(members)):
            family_pairs.append(tuple(sorted([members[i], members[j]])))

family_pairs = list(set(family_pairs))
print(f"Пар из family_id:     {len(family_pairs):,}")

# ── Позитивные пары из МПК ──
ipc_groups = defaultdict(list)
for pub_num, patent in patents.items():
    ipc = patent.get("ipc_codes", "")
    if ipc:
        # Берём первые 4 символа — подкласс (например "B03C")
        code = ipc.split(",")[0].strip()[:4]
        ipc_groups[code].append(pub_num)

ipc_pairs = []
for code, members in ipc_groups.items():
    if len(members) < 2:
        continue
    random.shuffle(members)
    # Не больше 50 пар на группу чтобы не перевзвешивать крупные классы
    count = 0
    for i in range(len(members)):
        for j in range(i + 1, len(members)):
            ipc_pairs.append(tuple(sorted([members[i], members[j]])))
            count += 1
            if count >= 50:
                break
        if count >= 50:
            break

ipc_pairs = list(set(ipc_pairs))
print(f"Пар из МПК:           {len(ipc_pairs):,}")